# SoDA: from a notebook to a working app

Train → Package → Deploy → Monitor

We use public handwritten digits to keep the model small and the deployment understandable. Run these cells in order with the Python 3.12 kernel.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "ml").exists():
    ROOT = ROOT.parent
assert (ROOT / "ml/train.py").exists(), "Open this notebook from the workshop repository"
sys.path.insert(0, str(ROOT))
import json

import joblib
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

from ml.train import split_data, train

## 1. Inspect the data

Each image has 8 × 8 grayscale pixels. Zero is black and 16 is bright. The target is the digit 0–9. This dataset ships with scikit-learn.

Source: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html

In [ ]:
digits = load_digits()
print("Examples:", len(digits.data))
print("Features:", digits.data.shape[1])
print("First label:", digits.target[0])
print(digits.images[0].astype(int))

## 2. Split before fitting preprocessing

The fixed seed gives everyone the same split. Stratification preserves class proportions. The test set stays separate while we fit the scaler and model.

In [ ]:
x_train, x_test, y_train, y_test = split_data()
print(f"Train: {len(x_train)}; test: {len(x_test)}")
pipeline = make_pipeline(
    MinMaxScaler(), SVC(C=10.0, probability=True, random_state=42)
)
pipeline.fit(x_train, y_train)

## 3. Evaluate on held-out examples

The confusion matrix uses actual labels as rows and predicted labels as columns. A good score here does not guarantee success on your own handwriting.

**Discuss:** Why does the test set need to stay outside `fit`?

The browser crops, centers, and averages drawings into 8 × 8 pixels before calling the API. That image preparation matters: shrinking the entire canvas made small drawings look unlike the training data. The nonlinear SVC handles these shapes better than our original linear baseline. Its highest-probability class can differ from `SVC.predict()`, so evaluation and serving both use `predict_proba().argmax()`.


In [ ]:
# Match the API: choose the class with the highest probability.
probabilities = pipeline.predict_proba(x_test)
predictions = pipeline.classes_[probabilities.argmax(axis=1)]
print(f"Held-out accuracy: {accuracy_score(y_test, predictions):.2%}")
print(confusion_matrix(y_test, predictions))

## 4. Package the complete pipeline

The reusable training command applies a 93% quality gate and saves the fitted preprocessing with the classifier. It also records the model checksum, split, evaluation, dependency versions, and input contract.

Only load joblib artifacts you created or trust. Loading one can execute Python code.

In [ ]:
metadata = train(ROOT / "artifacts")
print(
    json.dumps(
        {
            k: metadata[k]
            for k in ["model_version", "test_accuracy", "train_samples", "test_samples", "input"]
        },
        indent=2,
    )
)

## 5. Reload and predict

This is the same packaged pipeline the API loads on startup. Restart the API after replacing its artifact.

In [ ]:
loaded = joblib.load(ROOT / "artifacts/model.joblib")
assert (loaded.classes_[loaded.predict_proba(x_test).argmax(axis=1)] == predictions).all()
example = x_test[0:1]
print("Actual digit:", y_test[0])
print("Prediction:", loaded.classes_[loaded.predict_proba(example).argmax(axis=1)][0])
print("Class scores:", loaded.predict_proba(example).round(3))

## Your next step

In a terminal at the repository root, run:

```bash
bash scripts/start.sh
```

Open port 8000. Select a held-out example and click Predict. In a second terminal, run `python scripts/smoke.py`. Follow `docs/WORKSHOP.md` for validation, deployment, and monitoring exercises.

**Extension:** Try `C=0.1` as a candidate in a separate output directory. Before doing systematic tuning, add a validation split so you do not repeatedly optimize against the test set.